# NoProp Ternary — Lightning AI Free Studio — Experiment 1D

**Platform:** Lightning AI Free (T4, 15 credits/month)

## Parallel execution strategy

Open each file in a **separate browser tab** / platform session simultaneously:

| Platform | File | Assigned experiments | Est. time | Cost |
|---|---|---|---|---|
| Google Colab Free (T4) | `noprop_colab.ipynb` | 0A + 1A | ~12 min | Free |
| Kaggle Free — tab 1 (P100) | `noprop_kaggle.ipynb` | 0B + 1B | ~12 min | Free |
| Kaggle Free — tab 2 (P100) | `noprop_kaggle2.ipynb` | 0C + 1C | ~12 min | Free |
| Lightning AI Free (T4) | `noprop_lightning.ipynb` | 1D | ~11 min | Free (15 credits/month) |
| Kaggle Free 12 h session | `noprop_kaggle_5k.ipynb` | All 7 @ 5k steps | ~4-5 h | Free |

> **Note:** Lambda Labs has no free tier; SageMaker Studio Lab requires manual approval (>=1 business day). Both replaced by Kaggle parallel sessions above.


In [ ]:
# Lightning AI setup
# New Studio > Jupyter > GPU (T4)
# Free plan: 15 credits/month (~80 GPU hours)
import os, sys
RESULTS_ROOT = os.path.join(os.path.expanduser("~"), "drex_parallel_results")

import subprocess
subprocess.run(["pip", "install", "-q", "-U",
                "torch", "datasets", "transformers", "pandas", "matplotlib", "tqdm"], check=True)
print("Setup complete.")


In [ ]:
import time, random, math, json, platform as _platform
from pathlib import Path
import torch

PLATFORM  = "lightning"
SEED      = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

RUN_NAME = time.strftime("lightning_%Y%m%d_%H%M%S")
RUN_DIR  = Path(RESULTS_ROOT) / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
for d in [RUN_DIR, CKPT_DIR]: d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Platform :", PLATFORM)
print("Device   :", device)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU      : {p.name}  ({p.total_memory/1024**3:.1f} GB)")
print("Run dir  :", RUN_DIR)


In [ ]:
import os, sys, json, time, random, math, platform
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class LMDataset(Dataset):
    def __init__(self, token_ids, seq_len=128):
        self.seq_len = seq_len; self.tokens = token_ids
        self.n = max(0, (len(self.tokens) - 1) // self.seq_len)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        i = idx * self.seq_len
        return (torch.tensor(self.tokens[i:i+self.seq_len], dtype=torch.long),
                torch.tensor(self.tokens[i+1:i+self.seq_len+1], dtype=torch.long))

def build_wikitext2(seq_len=128, batch_size=16):
    ds_tr = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    ds_va = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
    toks = lambda ds: tokenizer("\n".join(ds["text"]), add_special_tokens=False)["input_ids"]
    return (DataLoader(LMDataset(toks(ds_tr), seq_len), batch_size=batch_size, shuffle=True,  drop_last=True),
            DataLoader(LMDataset(toks(ds_va), seq_len), batch_size=batch_size, shuffle=False, drop_last=False))

train_loader, val_loader = build_wikitext2()
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")


In [ ]:
# ── Quantization primitives ───────────────────────────────────────────────────
class TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, d):
        ctx.save_for_backward(w, d)
        return torch.where(w > d, torch.ones_like(w),
               torch.where(w < -d, -torch.ones_like(w), torch.zeros_like(w)))
    @staticmethod
    def backward(ctx, g):
        w, d = ctx.saved_tensors
        return g * (w.abs() <= 1.5 * d).to(g.dtype), None

def ternary_ste(w, d): return TernarySTE.apply(w, d)
def trust_scale(w, d): return (1.0 / (1.0 + (w.abs() - d).abs())).clamp(0.05, 1.0)
def stochastic_ternary_project(w, d):
    return (torch.rand_like(w) < (w.abs() / (d + 1e-8)).clamp(0, 1)).float() * w.sign()

class BitLinear(nn.Module):
    def __init__(self, in_f, out_f, mode="fp32", bias=True):
        super().__init__(); self.mode = mode
        # HESTIA init: scale=0.5 so softmax(logits/tau) immediately gives informative
        # non-zero expected weights; 0.02 produced near-uniform → effective weight ≈ 0
        if mode == "hestia": self.logits = nn.Parameter(torch.randn(out_f, in_f, 3) * 0.5)
        else: self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None
    def _d(self, w): return 0.7 * w.abs().mean().detach() + 1e-8
    def forward(self, x, tau=1.0):
        m = self.mode
        if   m == "fp32":   wq = self.weight
        elif m == "ste":    wq = ternary_ste(self.weight, self._d(self.weight))
        elif m == "trust":
            d = self._d(self.weight); wq = ternary_ste(self.weight, d) * trust_scale(self.weight, d)
        elif m == "dqt":    wq = stochastic_ternary_project(self.weight, self._d(self.weight))
        elif m == "hestia":
            states = torch.tensor([-1., 0., 1.], device=x.device).view(1, 1, 3)
            wq = (F.softmax(self.logits / max(tau, 1e-3), dim=-1) * states).sum(-1)
        else: raise ValueError(f"Unknown mode: {m}")
        return F.linear(x, wq, self.bias)

# ── Transformer blocks ────────────────────────────────────────────────────────
class TinyBlock(nn.Module):
    def __init__(self, d_model, n_heads, mode="fp32"):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        self.ff1 = BitLinear(d_model, 4*d_model, mode=mode)
        self.ff2 = BitLinear(4*d_model, d_model, mode=mode)
    def forward(self, x, tau=1.0):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        a, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + a)
        return self.ln2(x + self.ff2(F.gelu(self.ff1(x, tau=tau)), tau=tau))

class TinyNoPropTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_blocks=6, mode="fp32", max_len=256):
        super().__init__(); self.mode = mode; self.n_blocks = n_blocks
        self.tok = nn.Embedding(vocab_size, d_model); self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TinyBlock(d_model, n_heads, mode) for _ in range(n_blocks)])
        self.noise_proj = nn.ModuleList([BitLinear(d_model, d_model, mode="fp32") for _ in range(n_blocks)])
        self.head = BitLinear(d_model, vocab_size, mode="fp32")
    def base_embed(self, x):
        B, T = x.shape
        return self.tok(x) + self.pos(torch.arange(T, device=x.device).unsqueeze(0).expand(B, T))
    def forward_global(self, x, tau=1.0):
        h = self.base_embed(x)
        for blk in self.blocks: h = blk(h, tau=tau)
        return self.head(h)
    def forward_block_local(self, b, x, z_noisy, tau=1.0):
        h = self.base_embed(x).detach() + self.noise_proj[b](z_noisy.detach())
        out = self.blocks[b](h, tau=tau)
        return out, self.head(out)

def dead_neuron_rate(t): return (t.abs() < 1e-6).float().mean().item()


In [ ]:
def evaluate_ppl(model, val_loader, max_batches=50):
    model.eval(); losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(val_loader):
            if i >= max_batches: break
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model.forward_global(x).reshape(-1, len(tokenizer)), y.reshape(-1))
            losses.append(loss.item())
    model.train()
    m = sum(losses) / max(1, len(losses))
    return m, math.exp(min(m, 20.0))

def get_grad_norm(mod):
    sq = sum(float((p.grad.detach()**2).sum()) for p in mod.parameters() if p.grad is not None)
    return math.sqrt(max(sq, 0.0))

def save_ckpt(model, opt, step, tag, ckpt_dir):
    p = ckpt_dir / f"{tag}_step_{step:07d}.pt"
    torch.save({"model": model.state_dict(),
                "opt":   opt.state_dict() if not isinstance(opt, list) else [o.state_dict() for o in opt],
                "step":  step}, p)
    return str(p)


In [ ]:
def run_global(exp_id, mode="fp32", steps=800, lr=3e-4, eval_every=200, ckpt_dir=None,
               tau_decay=0.999, tau_floor=0.05, tau_init=1.5):
    vocab = len(tokenizer)
    model = TinyNoPropTransformer(vocab_size=vocab, mode=mode).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr)
    logs, it, tau = [], iter(train_loader), tau_init
    for step in tqdm(range(1, steps+1), desc=exp_id, leave=True):
        try: x, y = next(it)
        except StopIteration: it = iter(train_loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model.forward_global(x, tau=tau).reshape(-1, vocab), y.reshape(-1))
        loss.backward(); gn = get_grad_norm(model); opt.step()
        if mode == "hestia": tau = max(tau_floor, tau * tau_decay)
        if step % eval_every == 0 or step == 1:
            vl, vp = evaluate_ppl(model, val_loader)
            mem = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.
            logs.append({"exp_id":exp_id,"step":step,"train_loss":loss.item(),"val_loss":float(vl),
                         "val_ppl":vp,"grad_norm":gn,"dead_rate":0.,"max_mem_gb":mem,"tau":round(tau,4)})
    ckpt = save_ckpt(model, opt, steps, exp_id, ckpt_dir or CKPT_DIR)
    return model, pd.DataFrame(logs), ckpt

def run_noprop(exp_id, mode="ste", steps=800, lr=3e-4, eval_every=200, ckpt_dir=None,
               tau_decay=0.999, tau_floor=0.05, tau_init=1.5, ce_weight=0.5, clip_norm=1.0):
    vocab  = len(tokenizer)
    model  = TinyNoPropTransformer(vocab_size=vocab, mode=mode, n_blocks=6).to(device)
    shared = list(model.tok.parameters()) + list(model.pos.parameters()) + list(model.head.parameters())
    # FIX: block optimizers own only block-specific params;
    # shared params (tok/pos/head) stepped once per global step to prevent 6x conflicting updates
    bopts  = [torch.optim.AdamW(list(model.blocks[b].parameters())+list(model.noise_proj[b].parameters()), lr=lr)
              for b in range(model.n_blocks)]
    opt_shared = torch.optim.AdamW(shared, lr=lr)
    logs, it, tau = [], iter(train_loader), tau_init
    noise_sched = torch.linspace(1.0, 0.1, model.n_blocks, device=device)
    for step in tqdm(range(1, steps+1), desc=exp_id, leave=True):
        try: x, y = next(it)
        except StopIteration: it = iter(train_loader); x, y = next(it)
        x, y = x.to(device), y.to(device)
        y_emb = model.tok(y).detach()
        gnorms, drates, blosses = [], [], []
        opt_shared.zero_grad(set_to_none=True)
        for b in range(model.n_blocks):
            bopts[b].zero_grad(set_to_none=True)
            z = y_emb + noise_sched[b] * torch.randn_like(y_emb)
            out, logits = model.forward_block_local(b, x, z, tau=tau)
            alpha = 1.0 + 0.1 * (b / max(1, model.n_blocks-1))
            # FIX run3: CE only at last block (intermediate blocks are pure denoisers per NoProp paper)
            ce_term = (ce_weight * F.cross_entropy(logits.reshape(-1, vocab), y.reshape(-1))
                       if b == model.n_blocks - 1 else 0.0)
            loss = alpha * F.mse_loss(out, y_emb) + ce_term
            loss.backward()
            # FIX run3: clip_norm=None for HESTIA — clipping throttled Gumbel logit grads (run2: 14k->62k)
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    list(model.blocks[b].parameters()) + list(model.noise_proj[b].parameters()),
                    max_norm=clip_norm)
            gnorms.append(get_grad_norm(model.blocks[b])); drates.append(dead_neuron_rate(out.detach()))
            blosses.append(loss.item()); bopts[b].step()
        opt_shared.step()
        if mode == "hestia": tau = max(tau_floor, tau * tau_decay)
        if step % eval_every == 0 or step == 1:
            vl, vp = evaluate_ppl(model, val_loader)
            mem = torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.
            logs.append({"exp_id":exp_id,"step":step,"train_loss":sum(blosses)/len(blosses),
                         "val_loss":float(vl),"val_ppl":vp,"grad_norm":sum(gnorms)/len(gnorms),
                         "dead_rate":sum(drates)/len(drates),"max_mem_gb":mem,"tau":round(tau,4)})
    ckpt = save_ckpt(model, bopts[0], steps, exp_id, ckpt_dir or CKPT_DIR)
    return model, pd.DataFrame(logs), ckpt


In [ ]:
import time as _tm
_CALIB_N = 3
_vocab = len(tokenizer)
_mc = TinyNoPropTransformer(vocab_size=_vocab, mode="fp32").to(device)
_oc = torch.optim.AdamW(_mc.parameters(), lr=3e-4)
_xc, _yc = next(iter(train_loader)); _xc, _yc = _xc.to(device), _yc.to(device)
def _sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()
for _ in range(2):
    _oc.zero_grad(); F.cross_entropy(_mc.forward_global(_xc).reshape(-1,_vocab),_yc.reshape(-1)).backward(); _oc.step()
_sync()
_t0 = _tm.perf_counter()
for _ in range(_CALIB_N):
    _oc.zero_grad(); F.cross_entropy(_mc.forward_global(_xc).reshape(-1,_vocab),_yc.reshape(-1)).backward(); _oc.step()
_sync()
_g_sps = _CALIB_N / (_tm.perf_counter() - _t0)

_boc = [torch.optim.AdamW(list(_mc.blocks[b].parameters())+list(_mc.noise_proj[b].parameters())+list(_mc.tok.parameters())+list(_mc.pos.parameters())+list(_mc.head.parameters()),lr=3e-4) for b in range(_mc.n_blocks)]
_nc  = torch.linspace(1.0,0.1,_mc.n_blocks,device=device); _ye = _mc.tok(_yc).detach()
_t0 = _tm.perf_counter()
for _ in range(_CALIB_N):
    for b in range(_mc.n_blocks):
        _boc[b].zero_grad()
        _z = _ye + _nc[b]*torch.randn_like(_ye); _o,_l = _mc.forward_block_local(b,_xc,_z)
        (F.mse_loss(_o,_ye)+0.1*F.cross_entropy(_l.reshape(-1,_vocab),_yc.reshape(-1))).backward(); _boc[b].step()
_sync()
_n_sps = _CALIB_N / (_tm.perf_counter() - _t0)
del _mc, _oc, _boc, _xc, _yc, _ye

_hw = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ("Apple MPS" if hasattr(torch.backends,"mps") and torch.backends.mps.is_available() else "CPU")
print(f"Hardware : {_hw}")
print(f"Global   : {_g_sps:.2f} sps  ({1000/_g_sps:.0f} ms/step)")
print(f"NoProp   : {_n_sps:.2f} sps  ({1000/_n_sps:.0f} ms/step)")


In [ ]:
# ── Prior run summary ─────────────────────────────────────────────────────────
# 1D run1 2026-03-16: val_ppl=97506  (FAIL) -- tau_decay=0.999,  tau crossed 1.0 at step ~405
# 1D run2 2026-03-16: val_ppl=118912 (FAIL) -- tau_decay=0.9993, tau crossed 1.0 at step ~579
# Root cause: val_ppl spikes exactly when tau crosses 1.0 -- Gumbel-softmax sharpening
#   disrupts sequential block composition in forward_global. train_loss always falls.
# Root cause 3 (new): logit init * 0.02 -> softmax approx uniform -> weight ≈ 0 from step 1.
# Fix A: logit init * 0.02 -> * 0.5 (non-trivial initial soft ternary weights).
# Fix B: tau_init=0.5 -> tau stays in [0.3, 0.5] throughout 1200 steps; never crosses 1.0.

# ── Experiment definitions for this platform ─────────────────────────────────
# Set FORCE_RERUN = True to re-run even if a prior CSV exists in this RUN_DIR.
# Set FORCE_RERUN = False to resume from a crash (skip completed experiments).
FORCE_RERUN = True

EXPERIMENTS = [
    {
        "id": "1D_noprop_hestia",
        "kind": "noprop",
        "mode": "hestia",
        "steps": 1200,
        "desc": "NoProp + HESTIA Gumbel-soft ternary (tau_init=0.5, tau_decay=0.9998, tau_floor=0.3)",
        # Slower tau decay: tau stays >0.8 throughout 1200 steps (was 0.999 → tau=0.67 at 800)
        # Higher tau floor: prevents hard discretization during smoke test
        "tau_decay": 0.9998,
        "tau_floor": 0.3,
        "tau_init": 0.5,  # start with informative soft ternary (was 1.5 -> weights near-zero)
        "clip_norm": None,  # run3: no clip; clipping throttled Gumbel logit grads (14k->62k regression)
    }
]

# Gate: ppl < 40000 = model is at least improving from its ~50k starting ppl (learning confirmed)
# (Full convergence gate of 300 is for 50k+ step runs, not smoke tests)
GATE = {
    "1D_noprop_hestia": {
        "ppl": 40000,
        "dead": 0.3
    }
}

# ── Run loop (skip-if-done for crash resume; FORCE_RERUN overrides) ───────────
all_logs, results = [], []

for e in EXPERIMENTS:
    done_csv = RUN_DIR / f"{e['id']}_logs.csv"
    if done_csv.exists() and not FORCE_RERUN:
        print(f"SKIP  {e['id']} (already done — set FORCE_RERUN=True to re-run)")
        df = pd.read_csv(done_csv); all_logs.append(df); results.append(df.iloc[-1].to_dict())
        continue

    sps_est  = _n_sps if e["kind"] == "noprop" else _g_sps
    eta_sec  = e["steps"] / sps_est
    eta_mm, eta_ss = divmod(int(eta_sec), 60)
    print(f"\n{'-'*70}\nRUN  {e['id']}  ({e['kind']}/{e['mode']}  {e['steps']} steps  ETA~{eta_mm}m{eta_ss:02d}s)")
    print(f"     tau_decay={e.get('tau_decay',0.999)}  tau_floor={e.get('tau_floor',0.05)}\n{'-'*70}")

    t0 = time.time()
    if e["kind"] == "global":
        model, logs_df, ckpt = run_global(e["id"], mode=e["mode"], steps=e["steps"],
                                          tau_decay=e.get("tau_decay", 0.999),
                                          tau_floor=e.get("tau_floor", 0.05),
                                          tau_init=e.get("tau_init", 1.5))
    else:
        model, logs_df, ckpt = run_noprop(e["id"], mode=e["mode"], steps=e["steps"],
                                          tau_decay=e.get("tau_decay", 0.999),
                                          tau_floor=e.get("tau_floor", 0.05),
                                          tau_init=e.get("tau_init", 1.5),
                                          ce_weight=e.get("ce_weight", 0.5),
                                          clip_norm=e.get("clip_norm", 1.0))
    elapsed = time.time() - t0

    logs_df.to_csv(done_csv, index=False); all_logs.append(logs_df)
    last = logs_df.iloc[-1].to_dict()
    last.update({"elapsed_sec": round(elapsed,1), "checkpoint": ckpt,
                  "gate_ppl_pass":  bool(last["val_ppl"]  < GATE[e["id"]]["ppl"]),
                  "gate_dead_pass": bool(last["dead_rate"] < GATE[e["id"]]["dead"])})
    results.append(last)
    g = "PASS" if last["gate_ppl_pass"] and last["gate_dead_pass"] else "FAIL"
    print(f"DONE  {elapsed/60:.1f} min | val_ppl={last['val_ppl']:.1f} (gate<{GATE[e['id']]['ppl']}) | dead={last['dead_rate']:.3f} | tau_final={last.get('tau','?'):.3f} | {g}")

sumdf = pd.DataFrame(results).sort_values("val_ppl")
sumdf.to_csv(RUN_DIR / "summary.csv", index=False)
report = {"platform": PLATFORM, "run_name": RUN_NAME, "device": str(device),
           "seed": SEED, "experiments": results}
with open(RUN_DIR / "run_report.json", "w") as f: json.dump(report, f, indent=2)
print(f"\nAll done.  Results in: {RUN_DIR}")
print(f"Upload {RUN_DIR / 'run_report.json'} + {RUN_DIR / 'summary.csv'} to your shared folder.")
sumdf[["exp_id","val_ppl","dead_rate","elapsed_sec","gate_ppl_pass","gate_dead_pass"]]

In [ ]:
if all_logs:
    df = pd.concat(all_logs, ignore_index=True)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for eid, g in df.groupby("exp_id"):
        axes[0].plot(g["step"], g["val_ppl"],  label=eid)
        axes[1].plot(g["step"], g["dead_rate"], label=eid)
        axes[2].plot(g["step"], g["grad_norm"], label=eid)
    axes[0].set_title("Validation Perplexity"); axes[1].set_title("Dead Neuron Rate"); axes[2].set_title("Grad Norm")
    for ax in axes: ax.set_xlabel("step"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(RUN_DIR / "plots.png", dpi=110); plt.show()
    print("Plot saved:", RUN_DIR / "plots.png")
